# Session 34 SQL Subqueries

## SQL Subqueries

A **subquery** is a query nested inside another SQL query. The inner query executes first, and its result is passed to the outer query.

Subqueries help solve problems that require intermediate calculations, such as:

- Comparing values against an aggregate (MAX, MIN, AVG, etc.)
- Filtering records based on another query's result
- Performing multi-step data retrieval without creating temporary tables
- Writing cleaner and more readable SQL

**General Syntax:**

```sql
SELECT column_name
FROM table_name
WHERE column_name operator (
    SELECT ...
);
```

---

## Why Do We Need Subqueries?

Suppose we want to find the movie with the highest IMDb score.

One approach is to first find the maximum score.

```sql
SELECT MAX(score)
FROM movies;
```

Output (example):

| MAX(score) |
|------------|
| 9.3 |

Now, using that value, we manually write another query.

```sql
SELECT *
FROM movies
WHERE score = 9.3;
```

Although this works, it has several disadvantages:

- Requires executing two separate queries.
- The value (`9.3`) has to be manually copied.
- If the data changes, the query must be updated again.
- Cannot be automated.

A subquery eliminates all these problems.

```sql
SELECT *
FROM movies
WHERE score = (
    SELECT MAX(score)
    FROM movies
);
```

Execution order:

1. The inner query finds the maximum score.
2. The result is returned to the outer query.
3. The outer query retrieves all movies having that score.

---

## Types of Subqueries

Subqueries can be classified based on the number of values they return.

| Type | Returns |
|------|---------|
| Scalar Subquery | Exactly one value |
| Row Subquery | One row with multiple columns |
| Column Subquery | Multiple values from one column |
| Table Subquery | Multiple rows and multiple columns |
| Correlated Subquery | Executes once for every row processed by the outer query |

In this section, we will focus on **Independent Scalar Subqueries**.

---

## Independent (Non-Correlated) Subqueries

An **Independent** (or **Non-Correlated**) subquery executes only **once**.

The outer query waits for the subquery to finish execution, receives its result, and then executes.

Since the subquery does **not** depend on any row from the outer query, it can run independently.

---

## Scalar Subqueries

A **Scalar Subquery** returns exactly **one value** (one row and one column).

These are commonly used with operators such as:

- `=`
- `>`
- `<`
- `>=`
- `<=`
- `<>`

Example:

```sql
SELECT *
FROM movies
WHERE score = (
    SELECT MAX(score)
    FROM movies
);
```

The inner query returns a single value (`9.3`), making it a scalar subquery.

---

## Preparing the Dataset

Some of the following queries use the **gross** and **budget** columns for mathematical calculations.

If these columns contain empty strings, PostgreSQL cannot convert them into numeric values.

The following query converts both columns into the `DECIMAL` data type while safely replacing empty strings with `NULL`.

```sql
ALTER TABLE movies
ALTER COLUMN gross TYPE DECIMAL
USING NULLIF(TRIM(gross::TEXT), '')::DECIMAL,

ALTER COLUMN budget TYPE DECIMAL
USING NULLIF(TRIM(budget::TEXT), '')::DECIMAL;
```

---

## Example 1: Find the Movie with the Highest Profit

Profit can be calculated as:

```text
Profit = Gross - Budget
```

Using a scalar subquery:

```sql
SELECT *
FROM movies
WHERE (gross - budget) = (
    SELECT MAX(gross - budget)
    FROM movies
);
```

### How it works

The inner query calculates the highest profit.

```sql
SELECT MAX(gross - budget)
FROM movies;
```

Suppose it returns:

```text
2748.3
```

The outer query becomes:

```sql
SELECT *
FROM movies
WHERE (gross - budget) = 2748.3;
```

This returns the movie with the maximum profit.

---

### Alternative Approach using ORDER BY

The same problem can also be solved using sorting.

```sql
SELECT *
FROM movies
WHERE gross IS NOT NULL
AND gross <> 0
ORDER BY (gross - budget) DESC
LIMIT 1;
```

### Comparison

| Subquery | ORDER BY |
|----------|----------|
| Finds rows matching the maximum calculated value | Sorts every row before selecting one |
| Useful when comparing against aggregate values | Useful when ranking records |
| Easier to extend into more complex filtering | Often simpler for Top-N queries |

---

## Example 2: Count Movies with Above-Average Ratings

Find the number of movies whose IMDb score is greater than the average score.

```sql
SELECT COUNT(*)
FROM movies
WHERE score > (
    SELECT AVG(score)
    FROM movies
);
```

### Execution Flow

First, PostgreSQL calculates:

```sql
SELECT AVG(score)
FROM movies;
```

Suppose the average score is:

```text
6.8
```

The outer query becomes:

```sql
SELECT COUNT(*)
FROM movies
WHERE score > 6.8;
```

Finally, it returns the total number of movies with above-average ratings.

This demonstrates one of the most common applications of scalar subqueries—comparing values against an aggregate.

---

## Example 3: Find the Highest Rated Movie Released in 2000

Instead of finding the highest-rated movie overall, we first restrict the search to movies released in the year **2000**.

```sql
SELECT *
FROM movies
WHERE year = '2000'
AND score = (
    SELECT MAX(score)
    FROM movies
    WHERE year = '2000'
);
```

### Execution Flow

The inner query executes first.

```sql
SELECT MAX(score)
FROM movies
WHERE year = '2000';
```

Suppose it returns:

```text
8.8
```

The outer query then becomes:

```sql
SELECT *
FROM movies
WHERE year = '2000'
AND score = 8.8;
```

Only the highest-rated movie from the year 2000 is returned.

---

## Example 4: Highest Rated Movie Among Popular Movies

Find the highest-rated movie among movies whose number of votes is greater than the average number of votes.

```sql
SELECT *
FROM movies
WHERE score = (
    SELECT MAX(score)
    FROM movies
    WHERE votes > (
        SELECT AVG(votes)
        FROM movies
    )
);
```

This query contains **two nested scalar subqueries**.

### Execution Flow

**Step 1**

Calculate the average number of votes.

```sql
SELECT AVG(votes)
FROM movies;
```

Suppose it returns:

```text
320000
```

---

**Step 2**

Find the highest score among movies having more than 320000 votes.

```sql
SELECT MAX(score)
FROM movies
WHERE votes > 320000;
```

Suppose this returns:

```text
9.2
```

---

**Step 3**

Retrieve the movie having that score.

```sql
SELECT *
FROM movies
WHERE score = 9.2;
```

---

## Advantages of Subqueries

- Break complex problems into smaller logical steps.
- Avoid hardcoding intermediate values.
- Automatically adapt when data changes.
- Improve readability for aggregate-based filtering.
- Eliminate the need for temporary tables in many situations.

---

## Limitations of Subqueries

- Deeply nested subqueries can become difficult to understand.
- May be slower than joins for certain problems.
- Correlated subqueries can significantly impact performance on large datasets.
- Sometimes a `JOIN`, `CTE`, or `ORDER BY` solution is more efficient.

---

## Key Takeaways

- A subquery is a query inside another SQL query.
- The inner query executes before the outer query.
- Scalar subqueries return exactly one value.
- Independent (non-correlated) subqueries execute only once.
- Aggregate functions such as `MAX()`, `MIN()`, `AVG()`, and `COUNT()` are common inside scalar subqueries.
- Subqueries eliminate manual intermediate steps and make queries dynamic.
- Nested subqueries can solve multi-stage filtering problems elegantly.

## Independent Subquery – Row Subquery (One Column, Multiple Rows)

A **Row Subquery (One Column, Multiple Rows)** is an independent subquery that returns **multiple values from a single column**.

Unlike a **Scalar Subquery**, which returns only one value, this type of subquery returns a list of values. Therefore, it is commonly used with operators that can compare against multiple values, such as:

- `IN`
- `NOT IN`
- `ANY`
- `ALL`
- `EXISTS` (conceptually works with multiple rows)

**General Syntax:**

```sql
SELECT ...
FROM table_name
WHERE column_name IN (
    SELECT column_name
    FROM another_table
);
```

### Execution Flow

1. The inner query executes **once**.
2. It returns a list of values from a single column.
3. The outer query compares its rows against that list.

Since the inner query does not depend on the outer query, it is called an **Independent (Non-Correlated) Subquery**.

---

## Example 1: Find All Users Who Never Ordered

```sql
SELECT *
FROM users
WHERE user_id NOT IN (
    SELECT DISTINCT(user_id)
    FROM orders
);
```

### Execution Flow

**Step 1**

The subquery executes.

```sql
SELECT DISTINCT(user_id)
FROM orders;
```

Suppose it returns

```text
2
4
7
8
11
...
```

**Step 2**

The outer query becomes conceptually equivalent to

```sql
SELECT *
FROM users
WHERE user_id NOT IN (2,4,7,8,11,...);
```

Users whose IDs are not present in the orders table are returned.

---

## Example 2: Find All Movies Made by the Top 3 Directors

```sql
SELECT *
FROM movies
WHERE director IN (
    SELECT director
    FROM movies
    GROUP BY director
    ORDER BY SUM(gross) DESC NULLS LAST
    LIMIT 3
)
ORDER BY gross DESC NULLS LAST;
```

### Execution Flow

The subquery first finds the three directors whose movies generated the highest total gross revenue.

```sql
SELECT director
FROM movies
GROUP BY director
ORDER BY SUM(gross) DESC NULLS LAST
LIMIT 3;
```

Suppose it returns

```text
James Cameron
Christopher Nolan
Steven Spielberg
```

The outer query becomes

```sql
SELECT *
FROM movies
WHERE director IN (
    'James Cameron',
    'Christopher Nolan',
    'Steven Spielberg'
);
```

All movies directed by these three directors are returned.

Notice that the outer query **does not return only three movies**. It returns **every movie** made by those three directors.

---

## Example 3: Find Movies of Actors Having an Average Score Greater than 8.5

```sql
SELECT *
FROM movies
WHERE star IN (
    SELECT star
    FROM movies
    WHERE votes > 25000
    GROUP BY star
    HAVING AVG(score) > 8.5
);
```

### Execution Flow

The subquery performs the following operations:

- Ignores movies having fewer than 25,000 votes.
- Groups remaining movies by actor.
- Computes the average score of each actor's filmography.
- Returns actors whose average score exceeds 8.5.

Suppose it returns

```text
Tom Hanks
Leonardo DiCaprio
Christian Bale
```

The outer query then becomes

```sql
SELECT *
FROM movies
WHERE star IN (
    'Tom Hanks',
    'Leonardo DiCaprio',
    'Christian Bale'
);
```

Every movie featuring these actors is returned.

---

## Why Do We Need Subqueries?

One question naturally arises:

> Why can't we simply store the result of one query inside a variable and reuse it?

In traditional programming languages like Python, Java, or C++, we would simply write

```python
top_directors = ...
```

or

```python
average_score = ...
```

and reuse those variables wherever needed.

However, **SQL does not provide traditional user-defined variables inside ordinary queries**.

Instead, SQL solves this problem using:

- Subqueries
- Common Table Expressions (CTEs)
- Temporary tables
- Views

Subqueries therefore act as temporary, inline results that can immediately be consumed by another query.

This is also closely related to SQL being a **non-procedural (declarative) language**.

In procedural languages, the programmer specifies **how** to perform every step.

In SQL, the programmer specifies **what** result is required, while the database optimizer decides **how** to obtain it.

Because SQL is declarative rather than procedural, intermediate computations cannot simply be assigned to variables in the way they are in traditional programming languages. Subqueries bridge this gap by allowing one query's result to be used immediately inside another query.

---

## How Do You Recognize When a Subquery Can Be Used?

A good indicator is when solving the problem naturally requires **multiple logical steps**.

Ask yourself:

> "Do I first need to calculate something before I can answer the final question?"

If the answer is **yes**, then a subquery is often appropriate.

Typical examples include:

- Find employees earning more than the average salary.
- Find products costing more than the most expensive product in another category.
- Find users who never placed an order.
- Find movies made by the highest-grossing directors.
- Find records matching maximum or minimum values.

Whenever one query depends on the output of another query, a subquery is a strong candidate.

---

## Independent Subquery – Table Subquery (Multiple Columns, Multiple Rows)

A **Table Subquery** returns an entire table consisting of:

- Multiple columns
- Multiple rows

The outer query treats the returned result exactly like a temporary table.

These subqueries are commonly used with:

- Tuple comparisons
- `IN`
- `EXISTS`
- Joins
- CTEs

General form:

```sql
SELECT ...
FROM table_name
WHERE (column1, column2) IN (
    SELECT column1,
           column2
    FROM ...
);
```

---

## Example 1: Find the Most Profitable Movie of Each Year

```sql
SELECT *
FROM movies
WHERE (year, gross - budget) IN (
    SELECT year,
           MAX(gross - budget)
    FROM movies
    GROUP BY year
);
```

### Execution Flow

The subquery returns something conceptually like

| Year | Max Profit |
|------|-----------:|
|2018|950|
|2019|1120|
|2020|870|

The outer query checks whether each movie's

```text
(year, profit)
```

matches one of these tuples.

This is known as a **tuple comparison**.

---

## Example 2: Highest Rated Movie of Each Genre

```sql
SELECT *
FROM movies
WHERE (genre, score) IN (
    SELECT genre,
           MAX(score)
    FROM movies
    WHERE votes > 25000
    GROUP BY genre
);
```

The subquery returns

| Genre | Highest Score |
|--------|--------------:|
|Action|9.1|
|Comedy|8.7|
|Drama|9.2|

The outer query returns every movie matching these `(genre, score)` pairs.

---

## Example 3: Highest Grossing Movies of the Top Five Actor–Director Pairs

### Method 1 (Using a CTE)

```sql
WITH top_duos AS (
    SELECT star,
           director,
           MAX(gross) AS max_gross
    FROM movies
    GROUP BY star, director
    ORDER BY MAX(gross) DESC NULLS LAST
    LIMIT 5
)
SELECT *
FROM movies
WHERE (star, director, gross) IN (
    SELECT *
    FROM top_duos
);
```

The CTE behaves like a temporary table containing

| Star | Director | Max Gross |
|------|----------|----------:|

The outer query performs tuple matching against all three columns simultaneously.

---

### Method 2 (Using JOIN)

```sql
WITH top_5_pairs AS (
    SELECT star,
           director
    FROM movies
    WHERE star IS NOT NULL
      AND director IS NOT NULL
    GROUP BY star, director
    ORDER BY MAX(gross) DESC NULLS LAST
    LIMIT 5
)
SELECT m.*
FROM movies m
INNER JOIN top_5_pairs t
ON m.star = t.star
AND m.director = t.director
ORDER BY m.gross DESC NULLS LAST;
```

This solution performs the same task using a join instead of tuple matching.

On many databases, this style is often easier to optimize and extend.

---

# Correlated Subquery

A **Correlated Subquery** is a subquery that **depends on values from the outer query**.

Unlike independent subqueries, it **cannot execute on its own**.

Instead, the subquery executes **once for every row processed by the outer query**.

General form:

```sql
SELECT ...
FROM table1 t1
WHERE column > (
    SELECT ...
    FROM table2 t2
    WHERE t2.column = t1.column
);
```

Notice that the inner query refers to `t1`, which belongs to the outer query.

That dependency makes the subquery correlated.

---

## Example 1: Movies Scoring Above Their Genre Average

```sql
SELECT *
FROM movies m1
WHERE score > (
    SELECT AVG(score)
    FROM movies m2
    WHERE m2.genre = m1.genre
);
```

### Why Is This Correlated?

Inside the subquery,

```sql
m2.genre = m1.genre
```

references `m1`, which belongs to the outer query.

Therefore, the inner query cannot execute independently.

---

### Execution Flow

Assume the outer query begins scanning the movies table.

---

**Row 1**

Movie:

```text
Inception
Genre = Sci-Fi
Score = 8.8
```

The subquery becomes

```sql
SELECT AVG(score)
FROM movies
WHERE genre = 'Sci-Fi';
```

Suppose it returns

```text
8.1
```

The comparison becomes

```text
8.8 > 8.1
```

True.

"Inception" is returned.

---

**Row 2**

Movie:

```text
Titanic
Genre = Romance
Score = 7.9
```

Now the subquery executes **again**

```sql
SELECT AVG(score)
FROM movies
WHERE genre='Romance';
```

Suppose it returns

```text
7.4
```

Again

```text
7.9 > 7.4
```

True.

---

**Row 3**

Movie:

```text
Comedy Movie
Genre = Comedy
Score = 6.5
```

The subquery executes **again**

```sql
SELECT AVG(score)
FROM movies
WHERE genre='Comedy';
```

Suppose the average is

```text
7.2
```

Comparison

```text
6.5 > 7.2
```

False.

The movie is discarded.

This process continues **for every movie in the table**.

If there are 10,000 movies, the correlated subquery may execute approximately 10,000 times (though modern query optimizers may rewrite parts of the execution internally).

---

## Example 2: Find Every Customer's Favourite Food

```sql
WITH fav_food AS (
    SELECT name,
           f_name,
           COUNT(*) AS frequency
    FROM users t1
    JOIN orders t2
      ON t1.user_id = t2.user_id
    JOIN order_details t3
      ON t2.order_id = t3.order_id
    JOIN food t4
      ON t3.f_id = t4.f_id
    GROUP BY t1.name, t4.f_name
)
SELECT *
FROM fav_food f1
WHERE frequency = (
    SELECT MAX(frequency)
    FROM fav_food f2
    WHERE f2.name = f1.name
);
```

### Step 1

The CTE first creates a table similar to

| Customer | Food | Frequency |
|-----------|------|----------:|
|Alice|Pizza|5|
|Alice|Burger|3|
|Alice|Pasta|2|
|Bob|Burger|8|
|Bob|Pizza|5|
|Charlie|Pizza|2|

---

### Step 2

The outer query starts scanning this table.

Suppose the first row is

```text
Alice
Pizza
5
```

The correlated subquery becomes

```sql
SELECT MAX(frequency)
FROM fav_food
WHERE name='Alice';
```

It returns

```text
5
```

Comparison

```text
5 = 5
```

The row is returned.

---

Next row

```text
Alice
Burger
3
```

The subquery executes again

```sql
SELECT MAX(frequency)
FROM fav_food
WHERE name='Alice';
```

Returns

```text
5
```

Comparison

```text
3 = 5
```

False.

The row is discarded.

---

The process repeats for every customer until only the most frequently ordered food for each customer remains.

---

## Independent vs Correlated Subqueries

| Independent Subquery | Correlated Subquery |
|----------------------|---------------------|
| Executes once | Executes once per outer row |
| Can run independently | Depends on the outer query |
| Usually faster | Often slower |
| Returns result first | Recomputes result repeatedly |
| Easier to optimize | More computationally expensive |

---

# Using Subqueries with Different SQL Clauses

Subqueries are not limited to the `WHERE` clause. They can also appear inside:

- `SELECT`
- `FROM`
- `HAVING`

---

## Using Subqueries with SELECT

### Example 1: Percentage of Votes

```sql
SELECT name,
       (votes / (SELECT SUM(votes) FROM movies)) * 100
FROM movies;
```

The subquery computes the total number of votes.

Every row then divides its own vote count by this single value.

---

### Example 2: Display Genre Average Alongside Each Movie

```sql
SELECT name,
       genre,
       score,
       (
           SELECT AVG(score)
           FROM movies m2
           WHERE m2.genre = m1.genre
       ) AS avg_score_genre
FROM movies m1;
```

Each movie is displayed together with the average rating of its genre.

Since the subquery depends on `m1.genre`, it is a **correlated subquery**.

---

## Why Is Using Subqueries Inside SELECT Often Inefficient?

For every row returned by the outer query, the database may execute the subquery again.

For example, if the table contains

```text
100,000 movies
```

then the correlated subquery inside the `SELECT` list may execute approximately

```text
100,000 times
```

This repeated computation can become very expensive.

A better solution is often to use:

- `JOIN`
- Window functions (`OVER(PARTITION BY ...)`)
- Common Table Expressions (CTEs)

These approaches usually compute shared values only once, allowing the database optimizer to generate a much more efficient execution plan.

---

## Using Subqueries with FROM

A subquery inside the `FROM` clause behaves like a temporary table (also called a **derived table**).

Example:

```sql
SELECT r_name,
       avg_rating
FROM (
    SELECT r_id,
           AVG(restaurant_rating) AS avg_rating
    FROM orders
    GROUP BY r_id
) t1
JOIN restaurants t2
ON t1.r_id = t2.r_id;
```

Execution order:

1. The derived table calculates the average rating for every restaurant.
2. The outer query joins this temporary result with the `restaurants` table.
3. Restaurant names and their average ratings are displayed.

---

## Using Subqueries with HAVING

Example:

```sql
SELECT genre,
       AVG(score) AS avg_score_genre,
       (SELECT AVG(score) FROM movies) AS avg_score
FROM movies
GROUP BY genre
HAVING AVG(score) > (
    SELECT AVG(score)
    FROM movies
);
```

### Execution Flow

1. The subquery computes the overall average movie score.
2. The outer query groups movies by genre.
3. Each genre's average score is computed.
4. The `HAVING` clause compares each group's average against the overall average.
5. Only genres with above-average ratings are returned.

## Using Subqueries with INSERT, UPDATE, and DELETE

Subqueries are not limited to retrieving data using `SELECT`. They can also be used with Data Manipulation Language (DML) statements such as:

- `INSERT`
- `UPDATE`
- `DELETE`

Using subqueries with these statements allows one query to generate or identify the data that another query will insert, modify, or remove.

---

## Subquery with INSERT

A subquery inside an `INSERT` statement is used to insert the result of another query directly into a table.

Instead of manually specifying values, the inserted records are generated dynamically by the subquery.

### General Syntax

```sql
INSERT INTO table_name (column1, column2, ...)
SELECT ...
FROM ...
WHERE ...;
```

The `SELECT` statement acts as the source of the data being inserted.

---

## Example: Populate the `loyal_users` Table

Suppose we have already created the following table.

```sql
CREATE TABLE loyal_users (
    user_id INT PRIMARY KEY,
    name VARCHAR(50),
    expenditure INT
);
```

We want to insert only those customers who have placed **more than three orders**.

```sql
INSERT INTO loyal_users (user_id, name)
SELECT t1.user_id,
       t2.name
FROM orders t1
JOIN users t2
ON t1.user_id = t2.user_id
GROUP BY t1.user_id, t2.name
HAVING COUNT(*) > 3;
```

### Execution Flow

**Step 1**

The `SELECT` query executes first.

```sql
SELECT t1.user_id,
       t2.name
FROM orders t1
JOIN users t2
ON t1.user_id = t2.user_id
GROUP BY t1.user_id, t2.name
HAVING COUNT(*) > 3;
```

Suppose it returns

| user_id | name |
|---------:|------|
| 2 | Alice |
| 5 | Bob |
| 9 | Charlie |

---

**Step 2**

The returned rows are inserted into the `loyal_users` table.

Conceptually, SQL performs something similar to

```sql
INSERT INTO loyal_users(user_id, name)
VALUES
(2,'Alice'),
(5,'Bob'),
(9,'Charlie');
```

The advantage is that no manual insertion is required. The inserted rows are generated dynamically from another query.

---

## Subquery with UPDATE

A subquery inside an `UPDATE` statement is used to calculate the new value for one or more columns.

The new value may depend on:

- another table,
- aggregated data,
- or even the current row being updated.

This is one of the most common uses of **correlated subqueries**.

### General Syntax

```sql
UPDATE table_name
SET column_name = (
    SELECT ...
)
WHERE ...;
```

---

## Example: Calculate Loyalty Rewards

Suppose we want to give every loyal customer **10% cashback** based on their total spending.

```sql
UPDATE loyal_users
SET expenditure = (
    SELECT SUM(orders.amount) * 0.1
    FROM orders
    WHERE orders.user_id = loyal_users.user_id
);
```

> **Note:** In the original query, the column was written as `money`. Since the table definition contains the column `expenditure`, the `UPDATE` statement should update `expenditure`.

### Why Is This a Correlated Subquery?

Inside the subquery,

```sql
orders.user_id = loyal_users.user_id
```

references the outer table (`loyal_users`).

Therefore, the subquery depends on the current row being updated and executes once for each row.

---

### Execution Flow

Suppose the `loyal_users` table contains

| user_id | name |
|---------:|------|
|2|Alice|
|5|Bob|
|9|Charlie|

---

### Updating Alice

The outer query starts with

```text
user_id = 2
```

The subquery becomes

```sql
SELECT SUM(amount) * 0.1
FROM orders
WHERE user_id = 2;
```

Suppose Alice has spent

```text
₹18,000
```

The subquery returns

```text
1800
```

Alice's row becomes

| user_id | name | expenditure |
|---------:|------|------------:|
|2|Alice|1800|

---

### Updating Bob

The subquery executes again.

```sql
SELECT SUM(amount) * 0.1
FROM orders
WHERE user_id = 5;
```

Suppose Bob has spent

```text
₹12,500
```

The subquery returns

```text
1250
```

Bob's record is updated.

---

This process repeats for every customer in the table.

---

## Subquery with DELETE

Subqueries can also determine which rows should be removed.

Instead of manually specifying IDs, another query dynamically identifies the rows to delete.

### General Syntax

```sql
DELETE FROM table_name
WHERE column_name IN (
    SELECT ...
);
```

---

## Example: Delete Users Who Never Ordered

```sql
DELETE FROM users
WHERE user_id IN (
    SELECT user_id
    FROM users
    WHERE user_id NOT IN (
        SELECT DISTINCT(user_id)
        FROM orders
    )
);
```

### Execution Flow

**Step 1**

The innermost query executes.

```sql
SELECT DISTINCT(user_id)
FROM orders;
```

Suppose it returns

```text
1
2
3
5
7
```

These are the users who have placed at least one order.

---

**Step 2**

The middle query executes.

```sql
SELECT user_id
FROM users
WHERE user_id NOT IN (1,2,3,5,7);
```

Suppose it returns

```text
4
6
9
```

These users have never placed an order.

---

**Step 3**

Finally, the outer `DELETE` statement becomes conceptually equivalent to

```sql
DELETE FROM users
WHERE user_id IN (4,6,9);
```

These user records are permanently removed from the table.

---

## Why Is an Extra Subquery Needed in DELETE?

Many SQL database systems (including PostgreSQL and MySQL in certain scenarios) restrict deleting from a table while simultaneously reading from that same table in a way that may create ambiguity.

By nesting the query, SQL first computes the list of user IDs to be deleted and then performs the deletion, avoiding conflicts during execution.

---

## Verifying the Result

After the deletion, we can inspect the remaining users.

```sql
SELECT DISTINCT(user_id)
FROM users;
```

Only users who have placed at least one order should remain.

---

## Summary

| DML Statement | Purpose of the Subquery |
|--------------|-------------------------|
| `INSERT` | Generates the rows to be inserted into another table. |
| `UPDATE` | Computes new values dynamically for each row, often using correlated subqueries. |
| `DELETE` | Identifies which rows should be removed based on conditions or related tables. |

---

## Key Takeaways

- Subqueries can be used with all major DML statements: `INSERT`, `UPDATE`, and `DELETE`.
- An `INSERT` subquery acts as the data source for new records.
- An `UPDATE` subquery commonly calculates values dynamically for each row and is frequently correlated.
- A `DELETE` subquery allows rows to be removed based on complex conditions derived from other queries.
- Combining DML statements with subqueries enables powerful data manipulation while keeping the SQL concise and expressive.